In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql import *

In [0]:
data = [
    (1,"Apple"),
    (2,"Orange"),
    (3,"Banana"),
    (3,"Banana"),
    (1,"Orange"),
    (2,"Banana"),
    (1,"Banana"),
    (1,"Apple"),
    (4, "Apple")
]

df = spark.createDataFrame(data, ["id", "fruit"])


In [0]:
df.createTempView("customer_buy_fruit")

In [0]:
%sql 
drop table if exists customer_buy_fruit

In [0]:
%sql
select id, fruit, row_number() over (partition by id,fruit order by id) as rn from customer_buy_fruit

In [0]:
%sql
  select id from (select *,count(*) as fruit_count 
    from customer_buy_fruit group by id,fruit) tabl_1 
    group by id having count(id)>= 2

In [0]:
%sql
-- First  Approach
select id from (
  select id, fruit, row_number() over (partition by id,fruit order by id) as rn from customer_buy_fruit
) temp_table where rn = 1 group by id having count(id)>= 2;

-- Second Approach

  select id from (select *,count(*) as fruit_count 
    from customer_buy_fruit group by id,fruit) tabl_1 
    group by id having count(id)>= 2;

In [0]:
%sql
select * from interview_catalog.source.employee

In [0]:
%sql
-- List all employees working in the IT department.

select * from interview_catalog.source.employee where department = 'IT'

In [0]:
%sql
-- Find employees with salary greater than 80,000.
select * from interview_catalog.source.employee where salary > 80000

In [0]:
%sql
-- Display distinct departments.
select distinct(department) from interview_catalog.source.employee

In [0]:
%sql
-- List employees ordered by salary in descending order.
select * from interview_catalog.source.employee order by salary desc

In [0]:
%sql
-- Find employees who do not have a manager.
select * from interview_catalog.source.employee where manager_id is null;
    
-- Find the average salary of each department.
select department, avg(salary) as avg_salary from interview_catalog.source.employee group by department;
    
--Find total salary paid per department.
select department, sum(salary) as total_salary from interview_catalog.source.employee group by department;

In [0]:
%sql
-- Find the number of employees in each department.
select department, count(*) as total_count from interview_catalog.source.employee group by department;

In [0]:
%sql
-- Find departments having more than 3 employees.
select department from interview_catalog.source.employee group by department having count(*) > 3

In [0]:
%sql
--  Find the maximum salary in each department
-- select department, max(salary) as Maximum_Salary from interview_catalog.source.employee group by department;
                            

select * from (
  select  *,dense_rank() over(partition by department order by salary desc) as dr from interview_catalog.source.employee
) where dr = 1;

In [0]:
%sql
-- Find employees earning more than the company average salary.
select * from interview_catalog.source.employee
where salary > (select avg(salary) from interview_catalog.source.employee)

In [0]:
%sql
select emp_name, salary from interview_catalog.source.employee
where salary = (select max(salary) from interview_catalog.source.employee)

In [0]:
%sql
-- Find employees earning more than their department average.
select emp_name,department,salary from interview_catalog.source.employee e 
where salary > (select avg(salary) from interview_catalog.source.employee where e.department = department)
-- group by department, emp_name having salary > avg(salary)

In [0]:
%sql
-- Assign row numbers based on salary within each department.
select emp_name,department,salary, row_number() over(partition by department order by salary desc) as RN from interview_catalog.source.employee

In [0]:
%sql
-- Second Highest salary
select * from (
select *, dense_rank() over(order by salary desc) as RN from interview_catalog.source.employee
) where RN < 5

In [0]:
%sql
select * from (
  select *, dense_rank() over(partition by department order by salary desc) as DR from interview_catalog.source.employee
) where DR = 1

In [0]:
%sql
with dr_table as (
      select *, dense_rank() over(partition by department order by salary desc) as DR from interview_catalog.source.employee
)
select * from dr_table

In [0]:
data = [
(1,"Amit","IT",70000,"2023-01-10"),
(2,"Rahul","IT",80000,"2022-05-12"),
(3,"Neha","HR",60000,"2023-03-01"),
(4,"Priya","Finance",90000,"2021-07-15"),
(5,"Rohit","IT",75000,"2022-11-20"),
(6,"Ankit","HR",65000,"2023-02-10"),
(7,"Karan","Finance",85000,"2020-06-01")
]

columns = ["emp_id","name","department","salary","joining_date"]

df = spark.createDataFrame(data,columns)

In [0]:
    # IT department ke employees filter karo.

df.filter(col("department") == "IT").display()

In [0]:
# Ek new column add karo:
df.withColumn("salary_bonus", col("salary")* 0.15).display()

In [0]:
df.orderBy(col("salary").desc()).display()

In [0]:
print(df.count())

In [0]:
df.select(col("department")).distinct().display()

In [0]:
df.groupBy(col("department")).agg(avg(col("salary")).alias("avg_salary")).display()


In [0]:
win = Window.partitionBy(col("department")).orderBy(col("salary").desc())

df.withColumn("rank", dense_rank().over(win)).filter(col("rank") == 1).drop(col("rank")).display()

In [0]:
df.dropDuplicates(["emp_id"]).display()

In [0]:
df.withColumn("salary category",
              when(col("salary")> 80000, "High")
              .when(col("salary")> 60000, "Medium")
              .otherwise("Low")
              ).display()

In [0]:
df.groupBy(col("department")).count().alias("total_count").display()

In [0]:
wind = Window.partitionBy(col("department")).orderBy(col("salary").desc())

df.withColumn("rank", dense_rank().over(wind)).filter(col("rank") == 2).drop(col("rank")).display()

In [0]:
windo = Window.partitionBy(col("department")).rowsBetween(Window.unboundedPreceding, Window.currentRow).orderBy(col("emp_id"))

df.withColumn("rank", sum(col("salary")).over(windo)).display()

- emp_id
- emp_name
- department
- salary
- join_time

List of empployess whose salary is higher than the average salary of their department &
salary is higher than the salary of the previously joined employee in the same department and then 
rank them from highest to lowest salary

In [0]:
%sql
SELECT emp_id,
       emp_name,
       department,
       salary
       -- RANK() OVER(ORDER BY salary DESC) AS salary_rank
FROM (
        SELECT *,
               AVG(salary) OVER(PARTITION BY department) AS dept_avg_salary,
               coalesce(LAG(salary) OVER(PARTITION BY department ORDER BY hire_date),0) AS prev_salary
        FROM interview_catalog.source.employee
     ) t
WHERE salary > dept_avg_salary
AND salary > prev_salary;

In [0]:
%sql
update  interview_catalog.source.employee set salary = 120000,manager_id = 134 where emp_id =140

In [0]:
%sql
SELECT emp_id,
       emp_name,
       department,
       salary,
       RANK() OVER(ORDER BY salary DESC) AS salary_rank
FROM (
        SELECT *,
               AVG(salary) OVER(PARTITION BY department) AS dept_avg_salary,
               coalesce(LAG(salary) OVER(PARTITION BY department ORDER BY hire_date),0) AS prev_salary
        FROM interview_catalog.source.employee
     ) t
WHERE salary > dept_avg_salary
AND salary > prev_salary;

Find the Employees who earn more than their manager.

In [0]:
%sql
select * from interview_catalog.source.employee E1 left join interview_catalog.source.employee E2 on E1.manager_id = E2.emp_id where E1.salary > E2.salary

In [0]:
%sql
--  Give the difference between the salary and avg salary of each department for each employee
select * from (
    select *, avg(salary) over(partition by department) as avg_salary from interview_catalog.source.employee
) where salary > avg_salary order by department

In [0]:
%sql
-- Find the first employee who joined in each department.
select * from(
  select *, row_number() over(partition by department order by hire_date) as rn from interview_catalog.source.employee
) where rn = 1

In [0]:
%sql
-- Find employees whose salary is greater than both the previous and next employee in the same department.
select * from(
  select *, lag(salary) over(partition by department order by hire_date) as prev_salary,
  lead(salary) over(partition by department order by hire_date) as next_salary
  from interview_catalog.source.employee
) where salary > prev_salary and salary > next_salary
    


In [0]:
%sql
select * from interview_catalog.source.employee order by salary desc limit 5

In [0]:
%sql
-- Find employees who have the maximum salary difference from the previously joined employee in their department.
with diff_cte as (
  select *, abs(salary - coalesce(lag(salary) over(partition by department order by hire_date), 0)) as sal_diff
  from interview_catalog.source.employee
),
max_diff_cte as (
  select *, row_number() over(partition by department order by sal_diff desc) as sal_diff_rank
  from diff_cte where manager_id is not null
)

select e.*
from diff_cte e
join max_diff_cte m
  on e.emp_id = m.emp_id and sal_diff_rank = 1

In [0]:
%sql
-- Find top 3 highest paid employees in each department.
select * from (
  select *,
         dense_rank() over(partition by department order by salary desc) as salary_rank
  from interview_catalog.source.employee)
  where salary_rank <= 3

In [0]:
%sql
-- Find employees who joined after their manager.
select E.* from interview_catalog.source.employee E left join interview_catalog.source.employee E1 on E.manager_id = E1.emp_id where E.hire_date > E1.hire_date

In [0]:
%sql
select max(salary) from interview_catalog.source.employee

In [0]:
%sql
select department,max(emp_count) as max_emp from (select department, count(*) as emp_count from interview_catalog.source.employee group by department) group by department